# M62 — R0 → A2 distillation diagnosis

**Revision M62-2026-09-28-r1. Run sections 1–8 in order on a CUDA GPU. No phone needed.**

MonoDGP/M61 are parked. Teacher is R0 ResNet50 MonoDETR epoch185; student is A2 MobileNetV4 Medium MonoDETR epoch130. This notebook performs **zero training steps**. It checks native A2 inference/operator blockers, caches both frozen models on all 3,712 training images, and reports useful Vehicle geometry supervision. No validation-based selection and no automatic pilot.

Keep the same GPU type/software/output folder. After a restart rerun from section1; saved samples are verified and reused. Do not overwrite old cache identity files. Temporary NPZ read I/O errors get two retries; persistent errors report the path. All logs are saved to Drive.


## 1. Drive, paths, and durable logging

Check selection and dataset locations. Use a new M62 output folder; leave M61 artifacts untouched.


In [ ]:
# M62-2026-09-28-r1 — visible notebook revision.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import json, os, shlex, shutil, subprocess, sys, zipfile

print('M62-2026-09-28-r1')
DRIVE = Path('/content/drive/MyDrive')
MOBILE_REPO = Path('/content/mobile_adas3d')
REPO = Path('/content/MonoDETR_M62')
COMMIT = '6994b9f512400b258c6edb75f77423beb9c126f2'
OUTPUT = DRIVE / 'mobile_adas3d_outputs/students/monodetr_m62_r0_a2_diagnostic'
MANIFEST = OUTPUT / 'm62_manifest.json'
DATASET = Path('/content/kitti_m62')
SPLITS = DRIVE / 'mobile_adas3d_splits/kitti_chen'
R0_SELECTION = DRIVE / 'mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json'
A2_SELECTION = DRIVE / 'mobile_adas3d_outputs/students/monodetr_a2_gt/product_checkpoint_sweep/a2_product_selection.json'
DATASET_CANDIDATES = [Path('/content/kitti_m61'), Path('/content/kitti'),
                      Path('/content/monodetr_kitti'), DRIVE / 'datasets/kitti']

def run(command, name, cwd=None):
    command = [str(x) for x in command]
    log_path = OUTPUT / 'colab_logs' / (name + '.log')
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('+', shlex.join(command), '\nDurable log:', log_path, flush=True)
    tail = deque(maxlen=80)
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    with log_path.open('a', buffering=1) as log:
        log.write('\nSTART ' + datetime.now(timezone.utc).isoformat() + '\n')
        p = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
            tail.append(line.rstrip())
        rc = p.wait()
    if rc:
        raise RuntimeError(f'Exit {rc}; log={log_path}\n' + '\n'.join(tail))

def diagnostic(action, *args):
    run([sys.executable, '-u', MOBILE_REPO / 'scripts/diagnose_m62_r0_a2.py',
         action, '--manifest', MANIFEST, *args], action + '_' + '_'.join(args), MOBILE_REPO)

run(['nvidia-smi'], 'gpu')


## 2. Update main and prepare MonoDETR only

No MonoDGP checkout is created. Uses the pinned MonoDETR source, taxonomy and MobileNetV4 compatibility patches. Does not reset/delete an existing checkout. Builds the CUDA extension for this runtime.


In [ ]:
mobile_url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE_REPO.exists():
    run(['git', 'clone', '--branch', 'main', mobile_url, MOBILE_REPO], 'clone_mobile')
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=MOBILE_REPO, text=True).strip()
if branch != 'main':
    raise RuntimeError(f'Use main, not {branch!r}')
run(['git', 'pull', '--ff-only'], 'update_mobile', MOBILE_REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless',
     'numba', 'scikit-image', 'scikit-learn', 'tqdm', 'ninja', 'timm==1.0.20'], 'dependencies')
url = '/'.join(['https:', '', 'github.com', 'ZrrSkywalker', 'MonoDETR.git'])
if not REPO.exists():
    run(['git', 'clone', url, REPO], 'clone_monodetr')
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
if head != COMMIT:
    if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
        raise RuntimeError('Wrong upstream commit with local changes; use a new dedicated directory.')
    run(['git', 'fetch', 'origin', COMMIT], 'fetch_monodetr', REPO)
    run(['git', 'checkout', '--detach', COMMIT], 'pin_monodetr', REPO)
for patch in ('patch_monodetr_colab_compat.py', 'patch_monodetr_product_taxonomy.py',
              'patch_monodetr_mobilenetv4.py'):
    run([sys.executable, MOBILE_REPO / 'scripts' / patch, '--monodetr-repo', REPO], patch, MOBILE_REPO)
ops = REPO / 'lib/models/monodetr/ops'
patch_arch = ("import sys; from pathlib import Path; sys.path.insert(0, sys.argv[1]); "
              "from setup_m61_sources import patch_active_architecture; "
              "patch_active_architecture(Path(sys.argv[2]))")
run([sys.executable, '-c', patch_arch, MOBILE_REPO / 'scripts', ops / 'setup.py'], 'cuda_arch', MOBILE_REPO)
os.environ['MAX_JOBS'] = '2'
run([sys.executable, 'setup.py', 'build_ext', '--inplace'], 'cuda_build', ops)


## 3. Restore the original KITTI view

Original GT labels and calibrations are required. No cache predictions become GT. Train and validation ID hashes are checked, but **only train images are used in this notebook**.


In [ ]:
(DATASET / 'training').mkdir(parents=True, exist_ok=True)
(DATASET / 'ImageSets').mkdir(parents=True, exist_ok=True)
for name in ('image_2', 'label_2', 'calib'):
    alternatives = [f'training/{name}']
    if name in ('image_2', 'label_2'):
        alternatives.append(f'training/{name[:-1]}02')
    source = next((root / relative for root in DATASET_CANDIDATES for relative in alternatives
                   if (root / relative).is_dir()), None)
    if source is None:
        raise FileNotFoundError(f'Missing {name}; edit DATASET_CANDIDATES in section1.')
    link = DATASET / 'training' / name
    if not link.exists() and not link.is_symlink():
        link.symlink_to(source.resolve(), target_is_directory=True)
    elif link.resolve() != source.resolve():
        raise RuntimeError(f'Existing dataset link differs: {link}')
for split in ('train', 'val'):
    src, dst = SPLITS / (split + '.txt'), DATASET / 'ImageSets' / (split + '.txt')
    if dst.exists() and dst.read_bytes() != src.read_bytes():
        raise RuntimeError(f'Split changed: {dst}')
    shutil.copy2(src, dst)
for path in (R0_SELECTION, A2_SELECTION):
    if not path.is_file():
        raise FileNotFoundError(path)


## 4. Tests and exact-checkpoint provenance

Freezes R0 epoch185 and A2 epoch130 hashes, source, implementation, environment and policy. Existing incompatible output is preserved and rejected, not overwritten. No old runtime YAML is required.


In [ ]:
run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
     '-p', 'test_m62_diagnostic.py', '-v'], 'tests', MOBILE_REPO)
run([sys.executable, '-u', MOBILE_REPO / 'scripts/diagnose_m62_r0_a2.py', 'prepare',
     '--repo', REPO, '--dataset-root', DATASET, '--output', OUTPUT,
     '--r0-selection', R0_SELECTION, '--a2-selection', A2_SELECTION], 'prepare', MOBILE_REPO)


## 5. Bounded A2 native inference / compatibility inventory

5 warmups + 10 GPU model-only timings on one training image. Reports native custom attention modules and model size. **This is not Core ML conversion, a phone latency estimate, or an export pass.** No weight changes, optimizer steps or inference approximation.


In [ ]:
diagnostic('probe')


## 6. Cache the R0 teacher on all training images

ResNet50 here is intentional: it is the R0 teacher, not the A2 student. Both use identical unaugmented inputs and GT encoding, checked by hashes. Existing M61 caches are not transplanted.


In [ ]:
diagnostic('cache', '--role', 'teacher')


## 7. Cache A2 on the same training images

MobileNetV4 Medium + unchanged MonoDETR. Rerunning verifies saved files and predicts only missing samples. Keep the same GPU/environment.


In [ ]:
diagnostic('cache', '--role', 'student')


## 8. Diagnose and download — STOP before training

Vehicle score ≥0.30, 2D IoU ≥0.50, GT depth [2,60)m. Per component, count cases where teacher error is strictly below 95% of A2 error. A candidate needs ≥100 such objects and ≥25 in at least two distance bins (2–20,20–40,40–60m). Rank by improved-object count; tie order depth,center,dimensions,angle. Overall mean teacher superiority is **reported, not required**. These fixed screening thresholds are not safety requirements or proof KD will improve validation.

Return the ZIP for review whether a component is proposed or not. The paired 10-epoch pilot and numerical acceptance rules will be frozen separately before any training. MonoDGP/M61 remain parked.


In [ ]:
diagnostic('analyze')
report = json.loads((OUTPUT / 'm62_diagnostic.json').read_text())
print(json.dumps(report, indent=2))
bundle = OUTPUT / 'm62_results.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('m62_manifest.json', 'm62_a2_compatibility.json',
                 'm62_diagnostic.json', 'm62_geometry_pairs.json'):
        path = OUTPUT / name
        if not path.is_file():
            raise FileNotFoundError(path)
        z.write(path, name)
print('Return:', bundle, '\nSTOP: no training or phone deployment authorized.')
from google.colab import files
files.download(str(bundle))
